# SISAL-OSM Mapping: Complete Workflow

**Von SISAL-Datenbank zu funktionierender Overpass Query**

**Author:** Florian Thiery  
**Project:** EPICA-SISAL-FAIRification  
**Conference:** deRSE26

---

## Überblick

Dieser Workflow zeigt **Schritt-für-Schritt**, wie wir 305 SISAL Speleothem-Sites mit OpenStreetMap-Höhlen matchen.

```
SISAL CSV  →  WKT Parser  →  OSM Download  →  Local Matching  →  Overpass Query  →  RDF/Linked Data
   (305)        (lat/lon)      (70,815)         (232 matches)       (221 IDs)          (osm2lod)
```

---

## Schritt 1: SISAL-Daten laden

### Problem
SISAL speichert Koordinaten im **WKT-Format**: `POINT(lon lat)`

### Lösung
Regex-Parser extrahiert Latitude/Longitude

In [ ]:
import pandas as pd
import re
from pathlib import Path

# Load SISAL database
sisal_df = pd.read_csv('sisal_sites_all.csv')

print(f"📊 SISAL Database loaded: {len(sisal_df)} sites")
print(f"\nColumns: {list(sisal_df.columns)}")
print(f"\nExample WKT:")
print(sisal_df[['site_id', 'site_name', 'wkt']].head(3))

In [ ]:
# Parse WKT: POINT(lon lat) → (lat, lon)
def parse_wkt_point(wkt_string):
    """
    Extract coordinates from WKT POINT format.
    
    Input:  'POINT(11.67 47.08)'
    Output: (47.08, 11.67)  # (latitude, longitude)
    """
    match = re.search(r'POINT\(([+-]?\d+\.?\d*)\s+([+-]?\d+\.?\d*)\)', str(wkt_string))
    if match:
        lon = float(match.group(1))
        lat = float(match.group(2))
        return (lat, lon)
    return (None, None)

# Apply to all sites
sisal_df[['latitude', 'longitude']] = sisal_df['wkt'].apply(
    lambda x: pd.Series(parse_wkt_point(x))
)

print("✓ WKT parsed!\n")
print(sisal_df[['site_id', 'site_name', 'latitude', 'longitude']].head())

---

## Schritt 2: OSM-Höhlen herunterladen

### Problem
Overpass API hat bei vielen einzelnen Queries **Rate Limits** und **Timeouts**

### Lösung
**EINE** große Query für alle Höhlen weltweit, dann lokal matchen!

### Overpass Query (manuell ausgeführt)

```
[out:json][timeout:180];
(
  node["natural"="cave_entrance"];
  way["natural"="cave_entrance"];
);
out center;
```

**Ausführung:**
1. https://overpass-turbo.eu/ → Query einfügen → "Run"
2. Export → GeoJSON → Save as `osm_caves.geojson`
3. Datei neben dieses Notebook legen

In [ ]:
import json

# Load OSM data (GeoJSON from Overpass Turbo)
with open('osm_caves.geojson', 'r', encoding='utf-8') as f:
    osm_geojson = json.load(f)

# Extract cave features
osm_caves = []

for feature in osm_geojson.get('features', []):
    geometry = feature.get('geometry', {})
    properties = feature.get('properties', {})
    
    # Only Point geometries (nodes and way centers)
    if geometry.get('type') == 'Point':
        lon, lat = geometry['coordinates']
        
        osm_caves.append({
            'name': properties.get('name', 'Unnamed'),
            'osm_id': properties.get('id', properties.get('@id', 'unknown')),
            'lat': lat,
            'lon': lon
        })

print(f"✓ OSM caves loaded: {len(osm_caves):,} features\n")
print("Example caves:")
for cave in osm_caves[:5]:
    print(f"  • {cave['name']:<30} (ID: {cave['osm_id']})")

---

## Schritt 3: Matching-Algorithmus

### 3.1 Distanz berechnen (Haversine)

Great-circle distance zwischen zwei Koordinaten.

$$
d = 2r \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1) \cos(\phi_2) \sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)
$$

In [ ]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in kilometers."""
    R = 6371  # Earth radius in km
    
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)
    
    a = (math.sin(delta_lat / 2) ** 2 + 
         math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c

# Test
dist = haversine_distance(47.08, 11.67, 47.080068, 11.6714154)
print(f"Test: Spannagel cave → Spannagelhöhle = {dist:.3f} km ✓")

### 3.2 Namen vergleichen (String Similarity)

SequenceMatcher (Ratcliff/Obershelp) mit Normalisierung

In [ ]:
from difflib import SequenceMatcher

def string_similarity(s1, s2):
    """Calculate name similarity (0.0 to 1.0)."""
    if not s1 or not s2:
        return 0.0
    
    # Normalize: lowercase, remove "cave"/"höhle"
    s1_norm = s1.lower().replace(' cave', '').replace(' höhle', '').strip()
    s2_norm = s2.lower().replace(' cave', '').replace(' höhle', '').strip()
    
    return SequenceMatcher(None, s1_norm, s2_norm).ratio()

# Tests
tests = [
    ('Spannagel cave', 'Spannagelhöhle'),
    ('Sofular cave', 'Sofular Cave'),
    ('Bunker cave', 'Unnamed')
]

print("String Similarity Tests:\n")
for s1, s2 in tests:
    sim = string_similarity(s1, s2)
    print(f"  '{s1}' ↔ '{s2}': {sim:.3f}")

### 3.3 Combined Score

$$
\text{score} = 0.7 \times \text{name\_similarity} + 0.3 \times \text{proximity\_score}
$$

$$
\text{proximity\_score} = \max\left(0, 1 - \frac{\text{distance}}{\text{radius}}\right)
$$

**Weights:** 70% Name, 30% Proximity

---

## Schritt 4: Matching ausführen

Für jede SISAL-Site:
1. Finde OSM-Höhlen im 5km-Radius
2. Berechne Score für jeden Kandidaten
3. Wähle besten Match

In [ ]:
def match_sites(sisal_df, osm_caves, radius_km=5):
    """
    Match SISAL sites to OSM caves.
    
    Returns: DataFrame with match results
    """
    results = []
    total = len(sisal_df)
    
    print(f"Matching {total} SISAL sites to {len(osm_caves):,} OSM caves...")
    print(f"Search radius: {radius_km} km\n")
    
    for idx, sisal_row in sisal_df.iterrows():
        if pd.notna(sisal_row['latitude']) and pd.notna(sisal_row['longitude']):
            sisal_lat = sisal_row['latitude']
            sisal_lon = sisal_row['longitude']
            sisal_name = sisal_row['site_name']
            
            # Find candidates within radius
            candidates = []
            
            for osm_cave in osm_caves:
                distance = haversine_distance(
                    sisal_lat, sisal_lon,
                    osm_cave['lat'], osm_cave['lon']
                )
                
                if distance <= radius_km:
                    name_sim = string_similarity(sisal_name, osm_cave['name'])
                    
                    # Combined score
                    proximity_score = max(0, 1 - distance / radius_km)
                    combined_score = name_sim * 0.7 + proximity_score * 0.3
                    
                    candidates.append({
                        'osm_name': osm_cave['name'],
                        'osm_id': osm_cave['osm_id'],
                        'distance_km': round(distance, 3),
                        'name_similarity': round(name_sim, 3),
                        'score': round(combined_score, 3),
                        'lat': osm_cave['lat'],
                        'lon': osm_cave['lon']
                    })
            
            # Build result
            result = {
                'site_id': sisal_row['site_id'],
                'site_name': sisal_name,
                'latitude': sisal_lat,
                'longitude': sisal_lon,
                'has_match': len(candidates) > 0,
                'num_candidates': len(candidates)
            }
            
            if candidates:
                # Select best match
                best = max(candidates, key=lambda x: x['score'])
                result.update({
                    'osm_name': best['osm_name'],
                    'osm_id': best['osm_id'],
                    'distance_km': best['distance_km'],
                    'name_similarity': best['name_similarity'],
                    'score': best['score'],
                    'osm_lat': best['lat'],
                    'osm_lon': best['lon']
                })
            
            results.append(result)
        
        # Progress
        if (idx + 1) % 50 == 0:
            print(f"  {idx + 1}/{total}...")
    
    print(f"\n✓ Matching complete!\n")
    return pd.DataFrame(results)

# Execute matching
results_df = match_sites(sisal_df, osm_caves, radius_km=5)

---

## Schritt 5: Ergebnisse analysieren

In [ ]:
matched = results_df[results_df['has_match'] == True]
unmatched = results_df[results_df['has_match'] == False]

print("=" * 70)
print("MATCHING RESULTS")
print("=" * 70)

print(f"\nTotal SISAL sites: {len(results_df)}")
print(f"✓ Matched: {len(matched)} ({len(matched)/len(results_df)*100:.1f}%)")
print(f"✗ Unmatched: {len(unmatched)} ({len(unmatched)/len(results_df)*100:.1f}%)")

if len(matched) > 0:
    print(f"\nDistance Statistics (matched sites):")
    print(f"  Min:    {matched['distance_km'].min():.3f} km")
    print(f"  Max:    {matched['distance_km'].max():.3f} km")
    print(f"  Mean:   {matched['distance_km'].mean():.3f} km")
    print(f"  Median: {matched['distance_km'].median():.3f} km")
    
    print(f"\nName Similarity Statistics:")
    print(f"  Min:    {matched['name_similarity'].min():.3f}")
    print(f"  Max:    {matched['name_similarity'].max():.3f}")
    print(f"  Mean:   {matched['name_similarity'].mean():.3f}")
    print(f"  Median: {matched['name_similarity'].median():.3f}")

In [ ]:
# Top 10 Best Matches
print("\nTOP 10 BEST MATCHES:\n")
top10 = matched.nlargest(10, 'score')[[
    'site_name', 'osm_name', 'distance_km', 'name_similarity', 'score'
]]
top10

---

## Schritt 6: OSM-IDs extrahieren (für osm2lod)

### Problem
OSM-IDs können verschiedene Formate haben:
- `"node/123456"` (String mit Prefix)
- `"123456.0"` (Float als String)
- `123456` (Integer)

### Lösung
Robuster Parser der alle Formate erkennt

In [ ]:
# Extract OSM IDs as integers
osm_ids = []

for _, row in matched.iterrows():
    osm_id_str = str(row['osm_id']).strip()
    
    # Remove prefix if present (e.g., "node/123456" → "123456")
    if '/' in osm_id_str:
        osm_id_str = osm_id_str.split('/')[-1]
    
    # Convert to integer
    try:
        osm_id = int(float(osm_id_str))  # float() handles "123.0"
        osm_ids.append(osm_id)
    except ValueError:
        print(f"⚠️  Could not parse: {row['osm_id']}")

# Remove duplicates and sort
osm_ids = sorted(set(osm_ids))

print(f"✓ Extracted {len(osm_ids)} unique OSM node IDs\n")
print(f"First 10 IDs:")
for osm_id in osm_ids[:10]:
    print(f"  {osm_id}")

---

## Schritt 7: Overpass Query generieren

### Die finale Query die funktioniert! 🎉

Format:
```
[out:json][timeout:25];
(
  node(123456);
  node(789012);
  ...
);
out meta geom;
```

In [ ]:
# Build Overpass Query
query = "[out:json][timeout:25];\n(\n"

for osm_id in osm_ids:
    query += f"  node({osm_id});\n"

query += ");\nout meta geom;"

print("=" * 70)
print("FINAL OVERPASS QUERY")
print("=" * 70)
print(f"\nQuery contains {len(osm_ids)} OSM node IDs\n")
print("Preview (first 10 nodes):\n")
print("[out:json][timeout:25];")
print("(")
for osm_id in osm_ids[:10]:
    print(f"  node({osm_id});")
print(f"  ... ({len(osm_ids)-10} more nodes)")
print(");")
print("out meta geom;")

### Query testen

**Option 1: Overpass Turbo (Browser)**
1. Gehe zu: https://overpass-turbo.eu/
2. Kopiere Query oben
3. Klicke "Run"
4. → Siehst alle 221 Höhlen auf der Karte! 🗺️

**Option 2: API (Python)**

In [ ]:
from urllib.parse import urlencode

# Generate Overpass Turbo URL
overpass_turbo_url = "https://overpass-turbo.eu/?" + urlencode({
    'Q': query,
    'R': ''
})

print("🗺️  OVERPASS TURBO URL:\n")
print(overpass_turbo_url[:100] + "...")
print(f"\n(Full URL is {len(overpass_turbo_url)} characters)")

---

## Schritt 8: Ergebnisse speichern

In [ ]:
# Create output directory
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# 1. Save matching results
results_df.to_csv(output_dir / 'sisal_osm_matches.csv', index=False, encoding='utf-8-sig')
print(f"✓ Saved: output/sisal_osm_matches.csv")

# 2. Save Overpass Query
with open(output_dir / 'overpass_query.txt', 'w', encoding='utf-8') as f:
    f.write(query)
print(f"✓ Saved: output/overpass_query.txt")

# 3. Save Overpass Turbo URL
with open(output_dir / 'overpass_turbo_url.txt', 'w', encoding='utf-8') as f:
    f.write(overpass_turbo_url)
print(f"✓ Saved: output/overpass_turbo_url.txt")

# 4. Save OSM IDs (plain text, one per line)
with open(output_dir / 'osm_ids.txt', 'w', encoding='utf-8') as f:
    for osm_id in osm_ids:
        f.write(f"{osm_id}\n")
print(f"✓ Saved: output/osm_ids.txt")

# 5. Save OSM IDs (JSON)
import json
with open(output_dir / 'osm_ids.json', 'w', encoding='utf-8') as f:
    json.dump(osm_ids, f, indent=2)
print(f"✓ Saved: output/osm_ids.json")

print("\n🎉 All files saved successfully!")

---

## Zusammenfassung

### Workflow

```
1. SISAL CSV laden           → 305 Sites
2. WKT parsen                → (lat, lon)
3. OSM GeoJSON laden         → 70,815 Höhlen
4. Lokales Matching          → 232 Matches (76.1%)
5. OSM-IDs extrahieren       → 221 unique IDs
6. Overpass Query bauen      → Funktioniert! ✓
```

### Ergebnis

**Match-Rate:** 76.1% (232/305)
- Durchschnittliche Distanz: ~1.3 km
- Durchschnittliche Namen-Ähnlichkeit: ~0.47

### Warum funktioniert es?

1. ✅ **Keine API Rate Limits** (lokales Matching)
2. ✅ **Keine Timeouts** (eine große Query statt 305 kleine)
3. ✅ **Kombinierter Score** (70% Name + 30% Proximity)
4. ✅ **Robuste ID-Extraktion** (verschiedene Formate)

### Nächste Schritte

1. **osm2lod Pipeline**: `osm_ids.txt` verwenden für RDF-Export
2. **Manuelle Verifikation**: Top-20 wichtigste Sites checken
3. **Radius erweitern**: 10km für unmatched sites

---

**Author:** Florian Thiery  
**License:** MIT  
**Project:** EPICA-SISAL-FAIRification  